# Merge And Repair

This notebook merges the 4 first-pass batch outputs, repairs only the failed rows, and writes the final `submission.csv`.


## Drive And Repo Setup

Mount Drive, clone the repo into the runtime, and copy the required CSV inputs into the checkout.

- Reads: Google Drive, GitHub repo, runtime filesystem
- Writes: `/content/svg-kaggle-comptetition`, Drive output root
- Rerun-safe: Yes. It reclones the repo and recopies the inputs cleanly.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition-.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
OUTPUT_ROOT = DRIVE_ROOT / "svg-kaggle-comptetition" / "submission_outputs"

if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)
subprocess.check_call(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])

for required_name in ["test.csv", "train.csv", "sample_submission.csv"]:
    destination = CHECKOUT_PATH / required_name
    if destination.exists():
        continue

    preferred_sources = [
        DRIVE_ROOT / "svg-kaggle-comptetition" / required_name,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / required_name,
    ]
    copied = False
    for candidate in preferred_sources:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            copied = True
            break
    if not copied:
        for candidate in DRIVE_ROOT.rglob(required_name):
            if candidate.is_file():
                shutil.copy2(candidate, destination)
                copied = True
                break
    if not copied:
        raise FileNotFoundError(
            f"Could not find {required_name} in Google Drive. Copy it into {CHECKOUT_PATH} manually and rerun this cell."
        )

os.environ["SVG_KAGGLE_REPO_ROOT"] = str(CHECKOUT_PATH)
sys.path.insert(0, str(CHECKOUT_PATH))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repo checkout: {CHECKOUT_PATH}")
print(f"Drive output root: {OUTPUT_ROOT}")


## Package Check

Install any missing runtime packages required by the shared helper module and this notebook.

- Reads: Current Python environment
- Writes: Installed runtime packages
- Rerun-safe: Yes. It only installs missing packages.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "cairosvg": "cairosvg",
    "skimage": "scikit-image",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


## Load Helpers And Batch Artifacts

Import the shared helper module, load the analysis recommendation, and verify that all 4 batch artifacts exist before repair starts.

- Reads: Repo checkout, helper module, analysis recommendation, batch result CSV files
- Writes: In-memory merged batch tables only
- Rerun-safe: Yes. It only reloads artifacts.


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import SVG, Markdown, display

import submission_pipeline as pipeline

pipeline.set_global_seed(1337)
PATHS = pipeline.resolve_pipeline_paths(Path(os.environ["SVG_KAGGLE_REPO_ROOT"]), OUTPUT_ROOT)
CONFIG = pipeline.GenerationConfig(verbose_progress=True)
TEST_DF, SAMPLE_SUBMISSION_DF, TRAIN_DF = pipeline.load_competition_frames(PATHS)

print(f"Repo root: {PATHS.repo_root}")
print(f"Output root: {PATHS.output_root}")

ANALYSIS_RECOMMENDATION = pipeline.load_analysis_recommendation(PATHS)
BATCH_RESULTS_DF, MISSING_BATCHES = pipeline.load_all_batch_results(PATHS, batch_count=4)
if MISSING_BATCHES:
    raise RuntimeError(f"Missing batch results for batches: {MISSING_BATCHES}")
FIRST_PASS_SUCCESSES_DF, FAILED_DF = pipeline.split_success_and_failures(BATCH_RESULTS_DF)
print(f"Loaded first-pass successes: {len(FIRST_PASS_SUCCESSES_DF)}")
print(f"Loaded first-pass failures: {len(FAILED_DF)}")
display(pipeline.artifact_status_table(PATHS, batch_count=4))


## Load Model

Load the base model plus the LoRA adapter so the repair cell can retry only the failed rows.

- Reads: Base model on Hugging Face, LoRA adapter from repo
- Writes: In-memory model state only
- Rerun-safe: No. It is safe to rerun, but it reloads the model and costs time.


In [ ]:
RUNTIME = pipeline.load_lora_runtime(PATHS.adapter_dir)
print(f"Loaded runtime on: {RUNTIME.runtime_device}")


## Repair Failed Rows

Revisit only the failed rows using the fixed repair order and save the repaired-row artifacts into the merge directory.

- Reads: Failed first-pass rows, model runtime, analysis recommendation
- Writes: `merge/repaired_rows.csv`, `merge/remaining_failures.csv`
- Rerun-safe: Yes. It recomputes and overwrites the repair artifacts.


In [ ]:
REPAIRED_DF = pipeline.repair_failed_rows(
    FAILED_DF,
    RUNTIME,
    recommended_generation_mode=ANALYSIS_RECOMMENDATION["recommended_generation_mode"],
    config=CONFIG,
)
pipeline.write_repair_outputs(PATHS, REPAIRED_DF)
display(pipeline.summarize_repair_results(REPAIRED_DF))


## Assemble Final Submission

Merge successful first-pass rows and repaired rows back into sample-submission order, validate the result, and write the final `submission.csv`.

- Reads: Sample submission, first-pass successes, repaired rows
- Writes: `merge/submission.csv`
- Rerun-safe: Yes. It recomputes and overwrites the final submission artifact.


In [ ]:
FINAL_SUBMISSION_DF = pipeline.assemble_final_submission(
    SAMPLE_SUBMISSION_DF,
    FIRST_PASS_SUCCESSES_DF,
    REPAIRED_DF,
)
FINAL_VALIDATION_DF = pipeline.validate_submission_frame(FINAL_SUBMISSION_DF, SAMPLE_SUBMISSION_DF)
FINAL_SUBMISSION_PATH = pipeline.write_final_submission(PATHS, FINAL_SUBMISSION_DF)
print(f"Final submission path: {FINAL_SUBMISSION_PATH}")
print(f"Invalid SVG rows under the current gate: {int((~FINAL_VALIDATION_DF['valid']).sum())}")
display(FINAL_VALIDATION_DF.head())


## Merge Summary

Show final artifact status and a small sample of rows that required the zero-score fallback after repair.

- Reads: Merge directory artifacts
- Writes: Nothing
- Rerun-safe: Yes. Read-only display cell.


In [ ]:
display(pipeline.artifact_status_table(PATHS, batch_count=4))
ZERO_SCORE_DF = REPAIRED_DF[REPAIRED_DF["selected_strategy"] == "zero_score"].copy()
if ZERO_SCORE_DF.empty:
    print("No rows needed the zero-score fallback.")
else:
    display(ZERO_SCORE_DF[["id", "repair_strategy", "gate_error"]].head(20))
